In [57]:
import os, sys
import numpy as np
from collections import OrderedDict

sys.path.append('./official_github/')
from dataset.mnist import load_mnist

## 수치미분

In [58]:
def numerical_gradient(f, x):
    grad = np.zeros_like(x)
    
    for idx in range(x.size):
        tmp_val = x[idx]
        
        x[idx] = tmp_val + 1e-4
        fxh1 = f(x)
        
        x[idx] = tmp_val - 1e-4
        fxh2 = f(x)
        
        grad[idx] = (fxh1 - fxh2) / 2*(1e-4)
        
        x[idx]  = tmp_val
        
        return grad

## Activation Functions

+ sigmoid: $y = \frac{1}{1 + e^{-x}}$
  + 미분 = $\frac{e^{-x}}{(1 + e^{-x})^2} = \frac{1}{1+e^{-x}} \frac{e^{-x}}{1 + e^{-x}} = y(1-y)$
+ softmax: $\frac{e^{a_k}}{\Sigma_{i=1}^{n}{e^{a_i}}}$ = $\frac{e^{a_k} - C^\prime}{\Sigma_{i=1}^{n}{(e^{a_i} - C^\prime)}}$

In [59]:
def step_function(x):
    if x >= 0:
        return 1
    else: 
        return 0
    
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    if x >= 0:
        return x
    else:
        return 0

In [60]:
def softmax(x):
    return np.exp(x - np.max(x)) / np.sum(x - np.max(x))

## Loss function & Accuracy function
+ SSE = $\frac{1}{2}\Sigma_{k}{(y_k - t_k)^2}$
  + 1/2는 왜 하는거지? [미분할 때 더 편리하다!!](https://stats.stackexchange.com/questions/157398/what-is-the-purpose-of-the-convenience-1-2-fraction-on-the-sum-of-squared-erro)
+ Cross Entropy = $-\Sigma_{k}{t_k \log{y_k}}$

In [61]:
def mean_squared_error(y, t):
    return 0.5 * np.sum((y-t)**2)

def cross_entropy_error(y, t):
    return -np.sum(t * np.log(y + 1e7))

## Layers

In [62]:
class Relu:
    def __init__(self):
        self.mask = None
    
    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0
        return out
    
    def backward(self, dout):
        dout[self.mask] = 0
        dx = dout
        return dx

In [63]:
class Affine:
    def __init__(self, W, b) -> None:
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None
        
    def forward(self, x):
        self.x = x
        out = np.dot(x, self.W) + self.b
        return out
    
    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        return dx

In [64]:
class SoftmaxWithLoss:
    def __init__(self) -> None:
        self.loss = None
        self.y = None
        self.t = None
        
    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)
        return self.loss
    
    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        dx = (self.y - self.t) / batch_size
        return dx

## 오차역전파법 구현하기

In [65]:
class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size,
        weight_init_std=0.01):
        # 가중치 초기화
        self.params = {}
        self.params['W1'] = weight_init_std * \
            np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * \
            np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

        # 계층 생성
        self.layers = OrderedDict()
        self.layers['Affine1'] = \
            Affine(self.params['W1'], self.params['b1'])
        self.layers['Relu1'] = Relu()
        self.layers['Affine2'] = \
            Affine(self.params['W2'], self.params['b2'])
        self.lastLayer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)

        return x

    # x : 입력 데이터, t : 정답 레이블
    def loss(self, x, t):
        y = self.predict(x)
        return self.lastLayer.forward(y, t)

    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)
        if t.ndim != 1:
            t = np.argmax(t, axis=1)

        accuracy = np.sum(y == t) / float(x.shape[0])
        return accuracy

    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)

        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])

        return grads

    def gradient(self, x, t):
        # 순전파
        self.loss(x, t)

        # 역전파
        dout = 1
        dout = self.lastLayer.backward(dout)

        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        # 결과 저장
        grads = {}
        grads['W1'] = self.layers['Affine1'].dW
        grads['b1'] = self.layers['Affine1'].db
        grads['W2'] = self.layers['Affine2'].dW
        grads['b2'] = self.layers['Affine2'].db

        return grads

In [66]:
(x_train, t_train), (x_test, t_test) = \
        load_mnist(normalize=True, one_hot_label=True)

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

x_batch = x_train[:3]
t_batch = t_train[:3]

grad_numerical = network.numerical_gradient(x_batch, t_batch)
grad_backprop = network.gradient(x_batch, t_batch)

# 각 가중치의 차이의 절댓값을 구한 후, 그 절댓값들의 평균을 낸다.
for key in grad_numerical.keys():
    diff = np.average(np.abs(grad_backprop[key] - grad_numerical[key]))
    print(key + ":" + str(diff))

W1:0.009666200742398668
b1:0.07320810321613683
W2:0.2146178824155649
b2:5.570768720800542


## 오차 역전파로 학습하기

In [ ]:
(x_train, t_train), (x_test, t_test) = \
        load_mnist(normalize=True, one_hot_label=True)

iter_nums = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

for i in range(iter_nums):
        batch_mask = np.random.choice(train_size, batch_size)